# Municipalities

In [ ]:

# /Users/dmk6603/Documents/ransom/4-webapp/ransom/data/municipalities/municipalities_complete.csv
# Matchid,post_title,State,School?,Population,Goverment Service?,Likely Government/Municipality,Link,group_name,discovered,description,published,post_url,country,activity,website,duplicates
# 1,"Swansea, Massachusetts Police Department",MA,#N/D,"17,500",VERO,Likely Govt,https://www.swanseama.gov/departments/police_department/index.php,cryptolocker,00:00.0,,00:00.0,,US,Emergency Services,,[]
# 1,Amesbury Police Department,MA,#N/D,"17,366",,,https://www.amesburyma.gov/222/Police-Department,,,,,,,,,

# /Users/dmk6603/Documents/ransom/4-webapp/ransom/data/ranomware_live.csv
# post_title,group_name,discovered,published,website,country,activity,description,post_url
# waypointsolutions.com,dragonforce,2026-05-27T14:53:19.644042+00:00,2026-05-27T14:46:25.122450+00:00,waypointsolutions.com,US,Business Services,"Waypoint Business Solutions partners with Dell Technologies to provide comprehensive IT solutions, including hardware, software, and professional services.",http://z3wqggtxft7id3ibr7srivv5gjof5fwg76slewnzwwakjuf3nlhukdid.onion/blog/?post_uuid=3ff50f8d-524a-4e7e-93e8-da9955512a9e
# profundo.nl,dragonforce,2026-05-27T14:52:39.351466+00:00,2026-05-27T14:50:39.563058+00:00,profundo.nl,NL,Not Found,"Profundo is an independent research organization dedicated to advancing sustainable development and social justice through insightful, evidence-based research and advice.",http://z3wqggtxft7id3ibr7srivv5gjof5fwg76slewnzwwakjuf3nlhukdid.onion/blog/?post_uuid=cbfcb886-6c67-4cd9-baee-768296866a5c
# FWMK Law Offices,dragonforce,2026-05-27T13:53:12.067462+00:00,2026-05-27T13:32:34.153703+00:00,fwmk-law.co.il,IL,Business Services,"FWMK Law Offices is a full-service law firm known for its deep expertise in various legal areas, including high tech, mergers and acquisitions, and real estate. The firm provides top-quality legal advice and representation to a diverse clientele, including international clients, private investors, and public companies. With a team of highly skilled and responsive attorneys, FWMK emphasizes a professional yet friendly approach to client relations. Their commitment to excellence and understanding of business needs positions them as a trusted partner for organizations navigating the complexities of the legal landscape.",http://z3wqggtxft7id3ibr7srivv5gjof5fwg76slewnzwwakjuf3nlhukdid.onion/blog/?post_uuid=d9b7d0ae-881d-45da-8363-13538c9f623c

import pandas as pd
# merge both file on post_title and group_name
# print records in municipalities
municipalities = pd.read_csv('data/municipalities/municipalities_complete.csv')
# municipalities havin group name not empty
municipalities = municipalities[municipalities['group_name'].notna()]
ransomware_live = pd.read_csv('data/ranomware_live.csv')
merged = pd.merge(municipalities, ransomware_live, on=['post_title'], how='inner')

# count in municipalities records
print(f"Municipalities records: {len(municipalities)}")

# count in ransomware_live records
print(f"Ransomware Live records: {len(ransomware_live)}")

# count in merged records
print(f"Merged records: {len(merged)}")

# unique post_title in municipalities
print(f"Unique post_title in municipalities: {len(municipalities[['post_title']].drop_duplicates())}")




Municipalities records: 391
Ransomware Live records: 28533
Merged records: 357
Unique post_title in municipalities: 380


In [23]:
import pandas as pd
import re
from collections import defaultdict

municipalities = pd.read_csv('data/municipalities/municipalities_complete.csv')
ransomware_live = pd.read_csv('data/ranomware_live.csv')

# --- normalizzazione titoli ---

# uniforma apostrofi, spazi e case
def clean(text):
    text = str(text)
    for quote in ['\u2019', '\u2018', '\u0060', '\u00b4']:
        text = text.replace(quote, "'")
    return ' '.join(text.split()).lower()

# toglie il suffisso dopo la prima virgola: "alexander city, alabama" -> "alexander city"
def core_title(text):
    return clean(text).split(',')[0].strip()

# primo dominio presente nella stringa
domain_pattern = re.compile(r'[a-z0-9-]+(?:\.[a-z0-9-]+)+')
def extract_domain(text):
    found = domain_pattern.findall(str(text).lower())
    return found[0] if found else None

# --- costruzione indici da ransomware.live ---

# una sola riga per coppia (titolo, gruppo)
live = ransomware_live.drop_duplicates(subset=['post_title', 'group_name'])

strong = {}                    # (chiave_identita, group) -> date
by_identity = defaultdict(set) # chiave_identita -> insieme di date possibili

for _, r in live.iterrows():
    dates = (r['discovered'], r['published'])

    keys = {clean(r['post_title']), core_title(r['post_title'])}
    domain = extract_domain(r['website']) or extract_domain(r['post_title'])
    if domain:
        keys.add(domain)

    for k in keys:
        strong[(k, r['group_name'])] = dates
        by_identity[k].add(dates)

# --- funzione di match a due livelli ---

def find_dates(title, group):
    keys = {clean(title), core_title(title)}
    domain = extract_domain(title)
    if domain:
        keys.add(domain)

    # livello 1: identità + gruppo coincidono
    for k in keys:
        if (k, group) in strong:
            return strong[(k, group)]

    # livello 2: stessa identità ma gruppo diverso -> ok solo se la data è univoca
    possible = set()
    for k in keys:
        possible |= by_identity.get(k, set())
    if len(possible) == 1:
        return possible.pop()

    return (None, None)

# --- applica il match (solo ai comuni colpiti) ---

discovered_col = []
published_col = []
for _, m in municipalities.iterrows():
    if pd.isna(m['group_name']):
        discovered_col.append(None)
        published_col.append(None)
        continue
    disc, pub = find_dates(m['post_title'], m['group_name'])
    discovered_col.append(disc)
    published_col.append(pub)

municipalities['discovered'] = discovered_col
municipalities['published'] = published_col

# --- override manuali: comuni con nome irregolare ---
# mappa: titolo nel file municipalities -> titolo reale in ransomware.live
overrides = {
    'Midlothian, TX. Police Department':   'Midlothian Police Department',
    'City of Wheat Ridge':                 'Wheat Ridge County',
    'Ottawa County. OH':                   'co.ottawa.oh.us',
    'Rock County, Wisconsin':              'co.rock.wi.us',
    'Central SD 13J':                      'central.k12.or.us',
    'Okeene Elementary School Okeene, OK': 'Okeene Elementary School',
    'Plumas county.us':                    'plumascounty.us',
    'Hidalgo County.us':                   'hidalgocounty.us',
    'Harlingen TX.gov':                    'harlingentx.gov',
    'City of Clarksville':                 'cityofclarksville.com',
    'City of Shenandoah tx.us':            'shenandoahtx.us',
    'Groton Schools.org':                  'grotonschools.org',
    'Fulton Countyga.gov':                 'fultoncountyga.gov',
    "D'Hanis ISD":                         'dhanisisd.net',
    'MassDevelopment Boston, MA':          'MassDevelopment',
    'City of Defiance':                    'cityofdefiance.com',
}

# indice date per (titolo reale, gruppo) -> pesca la riga del gruppo giusto
dates_by_title_group = {}
for _, r in live.iterrows():
    dates_by_title_group[(r['post_title'], r['group_name'])] = (r['discovered'], r['published'])

for i, m in municipalities.iterrows():
    title = str(m['post_title']).strip()
    if title in overrides:
        key = (overrides[title], m['group_name'])
        if key in dates_by_title_group:
            disc, pub = dates_by_title_group[key]
            municipalities.at[i, 'discovered'] = disc
            municipalities.at[i, 'published'] = pub

# --- verifica e salvataggio ---

restored = municipalities['discovered'].notna().sum()
with_group = municipalities['group_name'].notna().sum()
print(f"date ripristinate: {restored} / {with_group} comuni con group_name")

municipalities.to_csv('data/municipalities/municipalities_complete.csv', index=False)

date ripristinate: 390 / 391 comuni con group_name


In [26]:
# --- 1. sostituisci VERO/FALSO con True/False ---
mappa_bool = {'VERO': True, 'FALSO': False, 'vero': True, 'falso': False}
municipalities = municipalities.replace(mappa_bool)

# --- 2. aggiungi la colonna hacked ---
municipalities['hacked'] = municipalities['group_name'].notna()

# --- 3. normalizza i nomi delle colonne ---
def normalize_col(name):
    name = name.strip().lower()
    name = name.replace('?', '')
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

municipalities.columns = [normalize_col(c) for c in municipalities.columns]

# --- 4. estrai il dominio dalla colonna link ---
from urllib.parse import urlparse

def extract_domain_from_link(url):
    if pd.isna(url) or str(url).strip() == '':
        return None
    netloc = urlparse(str(url).strip()).netloc.lower()
    if netloc.startswith('www.'):
        netloc = netloc[4:]
    return netloc if netloc else None

municipalities['domain'] = municipalities['link'].apply(extract_domain_from_link)

# --- verifica e salvataggio ---
print("colonne:", list(municipalities.columns))
print(municipalities[['link', 'domain']].head(10))

municipalities.to_csv('data/municipalities/municipalities_complete.csv', index=False)

colonne: ['matchid', 'post_title', 'state', 'school', 'population', 'goverment_service', 'likely_government_municipality', 'link', 'group_name', 'discovered', 'description', 'published', 'post_url', 'country', 'activity', 'website', 'duplicates', 'hacked', 'domain']
                                                link                    domain
0  https://www.swanseama.gov/departments/police_d...             swanseama.gov
1   https://www.amesburyma.gov/222/Police-Department            amesburyma.gov
2                 https://www.ci.durham.nh.us/police           ci.durham.nh.us
3  https://www.dover.nh.gov/government/city-opera...              dover.nh.gov
4               https://www.dicksoncountysheriff.org  dicksoncountysheriff.org
5                    https://www.cartercountytn.gov/        cartercountytn.gov
6     https://midlothian.tx.us/503/Police-Department          midlothian.tx.us
7             https://www.cityofovilla.org/66/Police          cityofovilla.org
8                     

In [27]:
# convert matchid to integer in municipalities_complete.csv
municipalities = pd.read_csv('data/municipalities/municipalities_complete.csv')
municipalities['matchid'] = municipalities['matchid'].fillna(0).astype(int)
municipalities.to_csv('data/municipalities/municipalities_complete.csv', index=False)

In [28]:
import pandas as pd

df = pd.read_csv("data/municipalities/municipalities_complete.csv")

df["population"] = df["population"].str.replace(",", "")
df["population"] = pd.to_numeric(df["population"], errors="coerce")
df.to_csv('data/municipalities/municipalities_complete.csv', index=False)

In [29]:
# remove description column
municipalities = pd.read_csv('data/municipalities/municipalities_complete.csv')
if 'description' in municipalities.columns:
    municipalities = municipalities.drop(columns=['description'])
municipalities.to_csv('data/municipalities/municipalities_complete.csv', index=False)   